In [2]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset

In [75]:
class CustomDropout_cnn_layers(nn.Module):
    def __init__(self, p=0.5):
        super().__init__()
        self.p = p

    def forward(self, x):
        if not self.training or self.p == 0:
            return x  # no dropout in eval mode
        
        N, C, H, W = x.shape
        mask = (torch.rand(N, C, 1, 1, device=x.device) > self.p).to(x.dtype)
        mask = mask / ((1 - self.p) + 1e-8)
        return x * mask


class CustomDropout_linear_layers(nn.Module):
    def __init__(self, p=0.5):
        super().__init__()
        self.p = p

    def forward(self, x):
        if not self.training or self.p == 0:
            return x  # no dropout in eval mode
        
        mask = (torch.rand(x.shape, device=x.device) > self.p).to(x.dtype)
        mask = mask / ((1 - self.p) + 1e-8)
        return x * mask
    
layer = torch.rand(4,5)
do = CustomDropout_linear_layers(p = 0.5)
do(layer)

tensor([[0.6908, 0.0871, 1.7761, 0.0000, 1.5444],
        [0.0000, 1.4512, 0.0000, 0.0000, 0.0000],
        [0.1696, 0.1327, 1.7932, 1.4286, 0.7996],
        [1.7101, 0.0000, 0.1886, 1.5326, 0.0000]])

In [ ]:
class VGG11(nn.Module):
    
    # TODO: add batchnorm

    def __init__(self, in_channels, num_classes=1000):
        super(VGG11, self).__init__()
        self.in_channels = in_channels
        self.num_classes = num_classes
        self.conv_layers = nn.Sequential(
            nn.Conv2d(self.in_channels, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.linear_layers = nn.Sequential(
            nn.Linear(in_features=512*7*7, out_features=4096),
            nn.ReLU(),
            CustomDropout_linear_layers(p=0.5),
            nn.Linear(in_features=4096, out_features=4096),
            nn.ReLU(),
            CustomDropout_linear_layers(p=0.5),
            nn.Linear(in_features=4096, out_features=self.num_classes)
        )
        
    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)
        x = self.linear_layers(x)
        return x
    
model = VGG11(in_channels = 3, num_classes = 20)
image_dummy = torch.rand(1, 3, 224, 224)
model(image_dummy)

tensor([[ 0.0152, -0.0166, -0.0143,  0.0076,  0.0104,  0.0025,  0.0091, -0.0026,
         -0.0095, -0.0133, -0.0033, -0.0030, -0.0041, -0.0111, -0.0066, -0.0197,
          0.0114,  0.0038, -0.0227,  0.0093]], grad_fn=<AddmmBackward0>)